In [ ]:
import pandas as pd
import s3fs
import arcticdb as adb
import datetime
import logging
from functools import reduce
import operator
import warnings
warnings.filterwarnings("ignore")
from py_vollib_vectorized.implied_volatility import vectorized_implied_volatility
from py_vollib_vectorized.greeks import delta, gamma, theta, vega, rho
from concurrent.futures import ThreadPoolExecutor, as_completed



class data_access:
    def __init__(self, server_ip = None, server_port= None, username=None, password=None):
        server_ip = "192.168.0.253" if server_ip is None else server_ip
        server_port = 9000 if server_port is None else server_port
        username = "minioadmin" if username is None else username
        password = "minioadmin" if password is None else password
        self.ac = adb.Arctic(f"s3://{server_ip}:data?port={server_port}&access={username}&secret={password}")
        if "map" not in self.ac.list_libraries():
            self.ac.create_library("map")
        map_lib = self.ac.get_library("map")
        self.map = []
        for data_type in ["equity", "option", "future", "index", "crypto"]:
            try:
                self.map.append(map_lib.read(f"map/{data_type}_map").data)
            except:
                logging.error(f"no map for {data_type} ")
        self.map = pd.concat(self.map, axis=0)
        pass

    def conditional_lookup(self, df, column, pattern):
        if not isinstance(pattern,str):
            return (df[column] == pattern)
        if pattern.endswith("*"):
            prefix = pattern[:-1]  # Remove the *
            return (df[column].str.startswith(prefix))
        elif pattern.startswith("*"):
            prefix = pattern[1:]  # Remove the *
            return (df[column].str.endswith(prefix))
        elif pattern.startswith(">="):
            if "-" in pattern:
                prefix = pattern[2:]  # Remove the >=
                prefix = pd.to_datetime(prefix)
            else:
                prefix = float(pattern[2:])  # Remove the >=
            return (df[column] >= prefix)
        elif pattern.startswith("<="):
            if "-" in pattern:
                prefix = pattern[2:]  # Remove the <=
                prefix = pd.to_datetime(prefix)
            else:
                prefix = float(pattern[2:])  # Remove the <=
            return (df[column] <= prefix)
        elif pattern.startswith(">"):
            if "-" in pattern:
                prefix = pattern[1:]  # Remove the >
                prefix = pd.to_datetime(prefix)
            else:
                prefix = float(pattern[1:])  # Remove the >
            return (df[column] > prefix)
        elif pattern.startswith("<"):
            if "-" in pattern:
                prefix = pattern[1:]  # Remove the <
                prefix = pd.to_datetime(prefix)
            else:
                prefix = float(pattern[1:])  # Remove the <
            return (df[column] < prefix)
        elif pattern.startswith("&"):
            prefix = pattern[1:]  # Remove the &
            if "-" in prefix:
                return (df[column].between(pd.to_datetime(prefix.split(",")[0]), pd.to_datetime(prefix.split(",")[1])))
            else:
                return (df[column].between(float(prefix.split(",")[0]), float(prefix.split(",")[1])))
        else:
            return (df[column] == pattern)
        
    def map_search(self, lookup, params={}):
        # print("map search",lookup,params)
        lookup = lookup.strip("/")
        data = self.map
        lookup_list = lookup.split("/")
        data_source = self.conditional_lookup(data,"source", lookup_list[1]) if len(lookup_list)>1 else True
        exch = self.conditional_lookup(data,"exch", lookup_list[2])  if len(lookup_list)>2 else True
        data_type = self.conditional_lookup(data,"data_type", lookup_list[3])  if len(lookup_list)>3 else True
        stock = self.conditional_lookup(data,"stock", lookup_list[4].upper())  if len(lookup_list)>4 else True
        
        if lookup_list[3] in ["Equity","Index"]:
            return data[data_source & exch & data_type & stock]
        else:
            data = data[data_source & exch & data_type & stock]
            if len(params)==0:
                return data
            extended_logic = []
            for key,value in params.items():
                extended_logic.append(self.conditional_lookup(data, key, value))
            extended_logic = reduce(operator.and_, extended_logic)
            # print(extended_logic)
            return data[extended_logic]
    
    def fetch_path(self,conn, path, timeframe="1t", start = None, end = None):
        try:
            q = adb.QueryBuilder()
            q = q.date_range((start, end)).resample(timeframe).agg({
                "open": "first",
                "high": "max",
                "low": "min",
                "close": "last",
                "volume": "sum",
                "oi": "last"
            })
            df = conn.read(path, query_builder=q).data
            return df
        except Exception as e:
            logging.warning(f"Skipping {path} due to error: {e}")
            return None
    
    def get_data(self, configs):
        dataset = {}
        if isinstance(configs,dict):
            configs = [configs]
        for config in configs:
            source = config["source"] if "source" in config else "basedata"
            exch = config["exch"] if "exch" in config else logging.error("mention exchange in all configs")
            data_type = config["data_type"] if "data_type" in config else logging.error("mention data_type in all configs")
            stock = config["stock"] if "stock" in config else logging.error("mention stock/keyword in all configs")
            timeframe = config["timeframe"] if "timeframe" in config else "1t"
            start = config["start"] if "start" in config else None
            end = config["end"] if "end" in config else None
            params = config["params"] if "params" in config else {}
            if isinstance(start,str):
                if  " " in start:
                    start = datetime.datetime.strptime(start,"%Y-%m-%d %H:%M:%S")
                else:
                    start = datetime.datetime.strptime(start,"%Y-%m-%d")
            if isinstance(end,str):
                if  " " in end:
                    end = datetime.datetime.strptime(end,"%Y-%m-%d %H:%M:%S")
                else:
                    end = datetime.datetime.strptime(end,"%Y-%m-%d")
            path = f"data/{source}/{exch}/{data_type}/{stock}"
            # print("path ",  path)
            if data_type not in self.ac.list_libraries():
                self.ac.create_library(data_type)
            lib = self.ac.get_library(data_type)
            # print("looking up ",path, params, config)
            for map in self.map_search(path,params).to_dict(orient="records"):
                # print(map)
                data = self.fetch_path(lib,map["path"],timeframe, start, end)
                if data is not None:
                    dataset[map["file"]] = data
        return pd.concat(dataset,axis=1),dataset
    
    def parse_option_metadata(self,ticker):
        # print("parse_option_metadata")
        try:
            parts = ticker.split("_")
            symbol = parts[0]
            expiry = datetime.datetime.strptime(parts[1], "%Y%m%d")
            opt_type = parts[2]
            strike = int(parts[3])
            return symbol, expiry, opt_type, strike
        except Exception as e:
            # print(f"[!] Failed parsing ticker: {ticker}, error: {e}")
            return None, None, None, None
        
    def process_ticker(self,ticker,df):
        if df.empty:
            # print(ticker,"no data on this ticker")
            return False
        if "CE" not in ticker and "PE" not in ticker:
            # print(ticker,"no option data on this ticker")
            return False
        df.index.name = "datetime"
        df.reset_index(inplace=True)

        symbol, expiry, opt_type, strike = self.parse_option_metadata(ticker)
        if None in (symbol, expiry, strike, opt_type):
            # print(f"⚠️ Skipping ticker {ticker} due to parsing failure.")
            return False

        df["ticker"] = ticker
        df["symbol"] = symbol
        df["expiry"] = expiry
        df["strike"] = strike
        df["type"] = opt_type

        df["date"] = df["datetime"].dt.date
        df["time"] = df["datetime"].dt.time
        df["dayofWeek"] = df["datetime"].dt.dayofweek

        df = df.sort_values("datetime").copy()
        df["AbsoluteMinute"] = df["datetime"].dt.hour * 60 + df["datetime"].dt.minute
        MARKET_OPEN_MINUTE = 9 * 60 + 15
        df["minute"] = df["AbsoluteMinute"] - MARKET_OPEN_MINUTE + 1
        df["minute"] = df["minute"].clip(lower=1, upper=376)
        df.drop(columns=["AbsoluteMinute"], inplace=True)
        df["days_to_expiry"] = (df["expiry"].dt.date - df["datetime"].dt.date).apply(lambda x: x.days)
        # df["Days_to_expiry"] = df["Days_to_expiry"].clip(lower=0)
        df["minutes_to_expiry"] = (df["expiry"] - df["datetime"]).dt.total_seconds() / 60

        self.dfs.append(df)

    def swap_format(self, data_dict):
        # print("swap_format")
        self.dfs = []
        with ThreadPoolExecutor(max_workers=16) as executor:
            futures = {executor.submit(self.process_ticker, ticker, df): ticker for ticker, df in data_dict.items()}
        # for future in as_completed(futures):
        #     result = future.result()
        #     if result is not None:
        #         dfs.append(result)
        # step 2 for options            
        # for ticker, df in data_dict.items():
        #     if "CE" not in ticker and "PE" not in ticker:
        #         continue
        #     df.index.name = "Datetime"
        #     df.reset_index(inplace=True)

        #     df.rename(columns={
        #         "open": "Open", "high": "High", "low": "Low",
        #         "close": "Close", "volume": "Volume", "oi": "Open Interest"
        #     }, inplace=True)

        #     symbol, expiry, opt_type, strike = self.parse_option_metadata(ticker)
        #     if None in (symbol, expiry, strike, opt_type):
        #         # print(f"⚠️ Skipping ticker {ticker} due to parsing failure.")
        #         continue

        #     df["Ticker"] = ticker
        #     df["Symbol"] = symbol
        #     df["Expiry"] = expiry
        #     df["Strike"] = strike
        #     df["Type"] = opt_type

        #     df["Date"] = df["Datetime"].dt.date
        #     df["Time"] = df["Datetime"].dt.time
        #     df["DayOfWeek"] = df["Datetime"].dt.dayofweek

        #     df = df.sort_values("Datetime").copy()
        #     df["AbsoluteMinute"] = df["Datetime"].dt.hour * 60 + df["Datetime"].dt.minute
        #     MARKET_OPEN_MINUTE = 9 * 60 + 15
        #     df["Minute"] = df["AbsoluteMinute"] - MARKET_OPEN_MINUTE + 1
        #     df["Minute"] = df["Minute"].clip(lower=1, upper=376)
        #     df.drop(columns=["AbsoluteMinute"], inplace=True)
        #     df["Days_to_expiry"] = ((df["Expiry"] - df["Datetime"]).dt.total_seconds() // (60 * 60 * 24) - 1).astype(int)
        #     df["Days_to_expiry"] = df["Days_to_expiry"].clip(lower=0)
        #     df["Minutes_to_expiry"] = (df["Expiry"] - df["Datetime"]).dt.total_seconds() / 60

        #     dfs.append(df)

        if not self.dfs:
            # print("No valid dataframes parsed. Returning empty DataFrame.")
            return pd.DataFrame()
        
        final_df = pd.concat(self.dfs, ignore_index=True)
        final_df.sort_values(["ticker", "datetime"], inplace=True)
        return final_df
    
    def compute_iv_and_greeks(self, df, r=0.1, model="black"):
        # print("compute_iv_and_greeks")
        df["expiry_with_time"] = pd.to_datetime(df["expiry"].dt.date.astype(str) + " 15:30:00")
        df["t"] = (df["expiry_with_time"] - df["datetime"]).dt.total_seconds() / (365 * 24 * 60 * 60)
        df.loc[df["t"]<=0,"t"]=0.00000001
        df["flag"] = df["type"].map({"CE": "c", "PE": "p"})

        required_cols = ["close", "underlying_close", "strike", "t", "flag"]
        df_valid = df.dropna(subset=required_cols)

        skipped_count = len(df) - len(df_valid)
        if skipped_count > 0:
            pass
            # print(f"⚠️ Skipped {skipped_count} rows due to missing required data for IV/Greeks calculation.")

        if df_valid.empty:
            # print("🚫 No rows available for Greek calculation.")
            return df


        df_valid["iv"] = vectorized_implied_volatility(df_valid["close"], df_valid["underlying_close"], df_valid["strike"], df_valid["t"], r, df_valid["flag"], model=model, return_as="series")
        df_valid["delta"] = delta(df_valid["flag"], df_valid["underlying_close"], df_valid["strike"], df_valid["t"], r, df_valid["iv"])
        df_valid["gamma"] = gamma(df_valid["flag"], df_valid["underlying_close"], df_valid["strike"], df_valid["t"], r, df_valid["iv"])
        df_valid["theta"] = theta(df_valid["flag"], df_valid["underlying_close"], df_valid["strike"], df_valid["t"], r, df_valid["iv"])
        df_valid["vega"]  = vega(df_valid["flag"],  df_valid["underlying_close"], df_valid["strike"], df_valid["t"], r, df_valid["iv"])
        df_valid["rho"]   = rho(df_valid["flag"],   df_valid["underlying_close"], df_valid["strike"], df_valid["t"], r, df_valid["iv"])

        for col in ["iv", "delta", "gamma", "theta", "vega", "rho"]:
            df[col] = df_valid[col]

        return df
    
    def add_atm_strike(self, df):
        # print("add_atm_strike")
        # Calculate the absolute difference between strike and underlying_close
        df["abs_diff"] = (df["strike"] - df["underlying_close"]).abs()

        # Find the minimum difference per timestamp
        min_diff = df.groupby("datetime")["abs_diff"].transform("min")

        # Mark the rows that are closest to ATM
        atm_mask = df["abs_diff"] == min_diff

        # Create ATM_strike by mapping Strike where the row is ATM
        df["atm_strike"] = df["strike"].where(atm_mask)

        # Forward-fill ATM_strike within each timestamp group
        df["atm_strike"] = df.groupby("datetime")["atm_strike"].transform("first")

        # Drop the temporary column
        df.drop(columns=["abs_diff"], inplace=True)

        return df


    def compute_straddle_price(self, df):
        # print("compute_straddle_price")
        ce_df = df[df["type"] == "CE"].rename(columns={"close": "CE_Close"})
        pe_df = df[df["type"] == "PE"].rename(columns={"close": "PE_Close"})
        join_cols = ["datetime", "strike", "expiry", "symbol"]
        merged = pd.merge(ce_df, pe_df, on=join_cols, suffixes=("_CE", "_PE"))
        merged["straddle_price"] = merged["CE_Close"] + merged["PE_Close"]
        return merged[["datetime", "strike", "expiry", "symbol", "straddle_price"]]

    def get_vertical_data(self, config):
        # print("get_vertical_data")
        config_option = config.copy()
        config_option["data_type"] = "Option"
        del config["params"]
        config_stock = config
        
        # print([config_stock,config_option])
        combined_df, dataset = self.get_data([config_stock,config_option])
        vertical_df = self.swap_format(dataset)

        underlying_key = next((key for key in dataset.keys() if "CE" not in key and "PE" not in key), None)
        if underlying_key:
            underlying_df = dataset[underlying_key].copy()
            underlying_df.index.name = "datetime"
            
            underlying_df = underlying_df.rename(columns={
                "open": "underlying_open",
                "high": "underlying_high",
                "low": "underlying_low",
                "close": "underlying_close",
                "volume": "underlying_volume",
                "oi": "underlying_oi"
            }) 
            # print(underlying_df)
            # print(vertical_df)               
            vertical_df = vertical_df.merge(underlying_df, on="datetime", how="left")
        else:
            pass
            # print("⚠️ No underlying data found to merge.")

        missing_underlying = vertical_df["underlying_close"].isna().sum()
        # print(f"ℹ️ Rows with missing Underlying_close after merge: {missing_underlying}")

        straddle_df = self.compute_straddle_price(vertical_df)
        vertical_df = vertical_df.merge(
            straddle_df, on=["datetime", "strike", "expiry", "symbol"], how="left"
        )
        vertical_df = self.compute_iv_and_greeks(vertical_df)
        vertical_df = self.add_atm_strike(vertical_df)
        vertical_df.drop(columns=["t", "flag"], inplace=True, errors="ignore")
        vertical_df["type"] = vertical_df["type"].map({"CE": 1, "PE": -1})
        return vertical_df


    

